# Serve NLS Model from Colab

Loads `adsabs/NLQT-Qwen3-1.7B` (LoRA adapter on Qwen3-1.7B) on a Colab T4 GPU
and exposes an OpenAI-compatible `/v1/chat/completions` endpoint via ngrok.

**Note:** NLQT-Qwen3.5-2B may echo natural language; NLQT-Qwen3-1.7B is trained on this repo's format and returns structured ADS queries.

**Steps:**
1. Run all cells
2. Copy the ngrok URL from the last cell
3. Set it as `NL_SEARCH_VLLM_ENDPOINT` in nectar's `.env.local`

**Requirements:** ngrok auth token (free at https://ngrok.com)

## 1. Check GPU & Install Dependencies

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# Use Colab's pre-installed torch to avoid torchvision "already registered" conflict.
# Restart runtime (Runtime → Restart session) after this cell, then run remaining cells.
# Both 1.7B and 2B were trained with Unsloth; transformers>=5.2.0 needed for Qwen3.5.
!pip install -q "transformers>=5.2.0" accelerate peft unsloth fastapi uvicorn pyngrok sentencepiece protobuf tiktoken

**Important:** After running the pip cell above, do **Runtime → Restart session** (or Restart runtime). Then run the cells below. This avoids the torchvision "already registered" conflict.

## 2. Load Model

In [ ]:
# Optional: authenticate with Hugging Face for higher rate limits
# Get a free token at https://huggingface.co/settings/tokens
import os

hf_token = os.environ.get("HF_TOKEN") 
if not hf_token:
    hf_token = input("Hugging Face token (or press Enter to skip): ")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print("Authenticated with Hugging Face")
else:
    print("Skipping auth - downloads may be slower")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_REPO = "adsabs/NLQT-Qwen3-1.7B"
CHECKPOINT = "checkpoint-2014"  # latest checkpoint
BASE_MODEL = "Qwen/Qwen3-1.7B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading tokenizer from {ADAPTER_REPO}/{CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, subfolder=CHECKPOINT, trust_remote_code=True)

print(f"Loading base model {BASE_MODEL}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map=DEVICE,
    trust_remote_code=True,
)

print(f"Applying LoRA adapter from {ADAPTER_REPO}/{CHECKPOINT}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO, subfolder=CHECKPOINT, torch_dtype=torch.float16)
model = model.merge_and_unload()

MODEL_NAME = ADAPTER_REPO  # for server endpoints
print(f"Model loaded and merged on {DEVICE}")

## 3. Quick Sanity Check

In [ ]:
import json

SYSTEM_PROMPT = '''Convert natural language to ADS/SciX search query. Output JSON: {"query": "..."}

Example:
User: Query: papers by Hawking on black holes from the 1970s
Date: 2025-03-16
Assistant: {"query": "author:\\"Hawking, S\\" abs:\\"black holes\\" pubdate:[1970 TO 1979]"}
Use author:, abs:, pubdate:, bibstem:, (inst: OR aff:), etc. Never echo the input.'''

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Query: papers about exoplanets by Seager from 2020\nDate: 2026-03-06"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Raw output: {response}")

# Parse thinking + JSON
if "</think>" in response:
    response = response.split("</think>")[-1].strip()
try:
    j = json.loads(response[response.find("{"):response.rfind("}")+1])
    print(f"Parsed query: {j.get('query', response)}")
except:
    print(f"Parsed: {response}")

## 4. Start FastAPI Server

In [ ]:
import json
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="NLS Colab Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

SYSTEM_PROMPT = '''Convert natural language to ADS/SciX search query. Output JSON: {"query": "..."}

Example:
User: Query: papers by Hawking on black holes from the 1970s
Date: 2025-03-16
Assistant: {"query": "author:\\"Hawking, S\\" abs:\\"black holes\\" pubdate:[1970 TO 1979]"}
Use author:, abs:, pubdate:, bibstem:, (inst: OR aff:), etc. Never echo the input.'''


class ChatMessage(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    model: str = "llm"
    messages: list[ChatMessage]
    max_tokens: int = 256
    temperature: float = 0.0
    chat_template_kwargs: dict = {}


def generate_query(messages, max_tokens=256):
    message_dicts = [{"role": m.role, "content": m.content} for m in messages]
    if not any(m["role"] == "system" for m in message_dicts):
        message_dicts.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
    prompt = tokenizer.apply_chat_template(message_dicts, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][prompt_tokens:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Strip thinking tags
    if "<think>" in response:
        parts = response.split("</think>")
        if len(parts) > 1:
            response = parts[-1].strip()

    # Extract JSON query
    try:
        json_start = response.find("{")
        json_end = response.rfind("}") + 1
        if json_start >= 0 and json_end > json_start:
            data = json.loads(response[json_start:json_end])
            response = data.get("query", response)
    except json.JSONDecodeError:
        pass

    return response, prompt_tokens, len(generated_ids)


@app.get("/health")
async def health():
    return {"status": "healthy", "model": MODEL_NAME, "device": DEVICE}


@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": "llm", "object": "model", "created": int(time.time()), "owned_by": "adsabs"}]}


@app.post("/v1/chat/completions")
async def chat_completions(request: ChatRequest):
    try:
        start = time.time()
        text, prompt_tok, comp_tok = generate_query(request.messages, request.max_tokens)
        elapsed = (time.time() - start) * 1000
        print(f"[{elapsed:.0f}ms] {text[:100]}")

        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": request.model,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": prompt_tok, "completion_tokens": comp_tok, "total_tokens": prompt_tok + comp_tok},
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("FastAPI app defined")

## 5. Expose via ngrok

Get a free auth token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import os

# Use env var if set (e.g. Colab secrets); otherwise input() works better than getpass in Cursor
NGROK_TOKEN = os.environ.get("NGROK_TOKEN") or input("Enter your ngrok auth token: ")

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

# Set up ngrok tunnel
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("=" * 60)
print(f"Public URL: {public_url}")
print(f"")
print(f"Add to nectar .env.local:")
print(f"NL_SEARCH_VLLM_ENDPOINT={public_url}/v1/chat/completions")
print("=" * 60)

# Run server using nest_asyncio-patched loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

## Usage

Once running, copy the `NL_SEARCH_VLLM_ENDPOINT` line above into your nectar `.env.local`, then:

```bash
cd ~/ads-dev/nectar
pnpm dev
```

Visit `http://localhost:8000/?nl_search=1` to enable the feature.

**Notes:**
- Free Colab sessions last ~12 hours before disconnecting
- The ngrok URL changes each time you restart
- Inference takes ~200-500ms per query on T4

**If you get natural language instead of structured queries:** Use the local Docker server instead—it has the hybrid NER pipeline that always returns structured ADS queries: `docker compose up` in this repo, then set `NL_SEARCH_PIPELINE_ENDPOINT=http://localhost:8000` and `NL_SEARCH_VLLM_ENDPOINT=http://localhost:8000/v1/chat/completions` in nectar.